# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — "The Freshness Multiplier" / "Old + Refreshed" (Finding #4 and #8)

The paper reports that 365+ day content refreshed within the last 30 days shows a 3.2x health-score
boost and 57x more impressions than similarly old, untouched content, and frames this as evidence
that "refresh works."

**My methodology question:** where does the "refreshed" label come from is it assigned at random,
or is it a page an editor already chose because it looked worth saving (existing backlinks, prior
traffic, brand priority)? If refresh is self-selected rather than randomly assigned, "page quality"
becomes a confounder: it drives both the decision to refresh AND the later performance. The
comparison as shown can't separate "refresh caused the lift" from "already-strong pages were the
ones refreshed." A cleaner test would need a matched-cohort or before/after-on-the-same-page design,
not a between-page comparison of refreshed vs. never-refreshed content.

### Finding 2 — ML Appendix "What Predicts Growth?" (logistic regression, 71% holdout accuracy)

The appendix reports content_age, days_since_update, and days_visible as the strongest signals
separating growing from declining pages.

**My methodology questions:**
1. **Window overlap:** the growing/declining label is built from the last-30-days-vs-prior-30-days
   impression trend. `days_visible` is described as a 90-day count. If that 90-day window includes
   the same last-30-days used to build the label, the feature partially already knows the answer
   (the "future/overlapping windows" leakage pattern) — the methodology page doesn't say whether the
   feature window was cut off before the label window starts.
2. **Split design:** the appendix says "80/20 split" but doesn't say whether it's a random row split
   or grouped by brand. With 57 brands in the portfolio, a random split can put the same brand's
   pages in both train and test, letting the model partly memorize brand identity rather than
   generalize exactly what a client-grouped split (like the one used in `w05_model.ipynb`) is
   meant to rule out.

Both questions are asked in the spirit the paper itself sets: it already flags several of its own
weak spots (survivor-bias caveats, unstable small buckets) this is the same discipline applied to
two spots the paper doesn't flag.

In [1]:
import pandas as pd

# Public numbers from the FlyRank research paper (docs/flyrank-seo-research-march-2026.pdf),
# restated here only as a reference point for the methodology questions above -- not derived
# from any client-level data, no computation needed for this section.
paper_findings_reference = pd.DataFrame([
    {
        'finding': 'Finding 1 -- Freshness Multiplier (365+ refreshed)',
        'reported_metric': 'health score boost',
        'reported_value': '3.2x (10.7 -> 34.5)',
    },
    {
        'finding': 'Finding 1 -- Freshness Multiplier (365+ refreshed)',
        'reported_metric': 'impression boost',
        'reported_value': '57x (71 -> 4039)',
    },
    {
        'finding': 'Finding 2 -- ML Appendix growth prediction',
        'reported_metric': 'logistic regression holdout accuracy',
        'reported_value': '71%',
    },
    {
        'finding': 'Finding 2 -- ML Appendix growth prediction',
        'reported_metric': 'top signals',
        'reported_value': 'content_age (negative), days_since_update, days_visible (positive)',
    },
])

print("=== Paper numbers referenced by my methodology questions above ===")
print(paper_findings_reference.to_string(index=False))
print("\nNo client data pulled in this section -- these are the paper's own published aggregates,")
print("restated here so Section 1's questions above stay traceable to a specific number.")

=== Paper numbers referenced by my methodology questions above ===
                                           finding                      reported_metric                                                     reported_value
Finding 1 -- Freshness Multiplier (365+ refreshed)                   health score boost                                                3.2x (10.7 -> 34.5)
Finding 1 -- Freshness Multiplier (365+ refreshed)                     impression boost                                                   57x (71 -> 4039)
        Finding 2 -- ML Appendix growth prediction logistic regression holdout accuracy                                                                71%
        Finding 2 -- ML Appendix growth prediction                          top signals content_age (negative), days_since_update, days_visible (positive)

No client data pulled in this section -- these are the paper's own published aggregates,
restated here so Section 1's questions above stay traceable to a spe

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model (`w05_model.ipynb`) already used a client-grouped split, not a naive random one
so the "before" I'm reproducing here is the naive version I *didn't* run in Week 5: a plain random
row split, where the same client's pages can land in both train and test. Same query, same features,
same label, same decision day (2026-03-15) as Week 5 only the split strategy changes. This isolates
exactly what the grouped split is protecting against.

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DECISION_DAY = '2026-03-15'
RANDOM_STATE = 42
K = 50

# --- Same feature + label query as w05_model.ipynb (unchanged on purpose) ------------
features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_clicks ELSE 0 END)      AS clk_trailing,
            AVG(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_avg_position END)       AS pos_trailing,
            MAX(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END)                            AS has_ga4_data
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT
        fx.*,
        DATE_DIFF('day', dc.content_created_date, DATE '{DECISION_DAY}') AS content_age_days
    FROM fx
    JOIN read_parquet('{REL}/dim_content.parquet') dc
        ON fx.content_hash_id = dc.content_hash_id
""").df()

labels = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date > DATE '{DECISION_DAY}' THEN gsc_impressions ELSE 0 END) AS imp_after
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
""").df()

merged = features.merge(labels, on=['client_hash_id', 'content_hash_id'])
merged['is_declining'] = (merged['imp_after'] < 0.8 * merged['imp_trailing']).astype(int)
merged['ctr_trailing'] = merged['clk_trailing'] / merged['imp_trailing'].replace(0, pd.NA)

scored = merged[merged['imp_trailing'] > 0].copy()
scored['ctr_trailing'] = scored['ctr_trailing'].astype(float)

FEATURE_COLS = ['imp_trailing', 'clk_trailing', 'pos_trailing', 'ctr_trailing', 'has_ga4_data', 'content_age_days']
GROUP_COL = 'client_hash_id'


def precision_at_k(labels_sorted_desc, k):
    return np.asarray(labels_sorted_desc)[:k].mean()


def run_split(train_idx, test_idx, split_name):
    train_df = scored.iloc[train_idx].copy()
    test_df = scored.iloc[test_idx].copy()

    X_train, y_train = train_df[FEATURE_COLS].values, train_df['is_declining'].values
    X_test, y_test = test_df[FEATURE_COLS].values, test_df['is_declining'].values

    rf = RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    test_df['score_rf'] = rf.predict_proba(X_test)[:, 1]

    ranked = test_df.sort_values('score_rf', ascending=False)
    p_at_k = precision_at_k(ranked['is_declining'].values, K)

    train_clients = set(train_df[GROUP_COL])
    test_clients = set(test_df[GROUP_COL])
    overlap = train_clients & test_clients

    return {
        'split': split_name,
        f'precision_at_{K}': round(p_at_k, 3),
        'base_rate': round(test_df['is_declining'].mean(), 3),
        'n_test_rows': len(test_df),
        'n_test_clients': test_df[GROUP_COL].nunique(),
        'client_overlap_train_test': len(overlap),
    }


# --- BEFORE: naive random row split -- ignores that rows repeat by client -----------
random_splitter = ShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
rand_train_idx, rand_test_idx = next(random_splitter.split(scored))
result_random = run_split(rand_train_idx, rand_test_idx, 'BEFORE: random row split')

# --- AFTER: client-grouped split -- same as w05_model.ipynb -------------------------
group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
grp_train_idx, grp_test_idx = next(group_splitter.split(scored, groups=scored[GROUP_COL]))
result_grouped = run_split(grp_train_idx, grp_test_idx, 'AFTER: client-grouped split')

comparison = pd.DataFrame([result_random, result_grouped])
print("=== Before/after: split strategy comparison (Random Forest, same features/label) ===")
print(comparison.to_string(index=False))
print(f"\nGap in precision@{K}: {result_random[f'precision_at_{K}'] - result_grouped[f'precision_at_{K}']:+.3f}")
print("If BEFORE's client_overlap_train_test > 0, some of BEFORE's score may be the model")
print("recognizing a client it already saw in training, not genuine out-of-client skill.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Before/after: split strategy comparison (Random Forest, same features/label) ===
                      split  precision_at_50  base_rate  n_test_rows  n_test_clients  client_overlap_train_test
   BEFORE: random row split             0.78      0.326        30397              43                         43
AFTER: client-grouped split             0.56      0.373        13210               9                          0

Gap in precision@50: +0.220
If BEFORE's client_overlap_train_test > 0, some of BEFORE's score may be the model
recognizing a client it already saw in training, not genuine out-of-client skill.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.